In [ ]:
!pip install unsloth
!pip install rouge-score bert-score datasets accelerate trl peft bitsandbytes
!pip install datasets==3.6.0
import datasets
dataset = datasets.load_dataset('csebuetnlp/xlsum','serbian_latin')


In [ ]:

print(f"Train: {len(dataset['train'])}")
print(f"Valid: {len(dataset['validation'])}")
print(f"Test:  {len(dataset['test'])}")

ex = dataset["train"][0]
print("\nTitle:", ex["title"])
print("Article:", ex["text"][:300], "...")
print("Summary:", ex["summary"])


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def filter_by_length(example):
    article_words = len(example["text"].split())
    summary_words = len(example["summary"].split())
    return 50 < article_words < 3000 and 10 < summary_words < 300

dataset_filtered = dataset.filter(filter_by_length)
print(f"После фильтрации — Train: {len(dataset_filtered['train'])}, "
      f"Valid: {len(dataset_filtered['validation'])}, "
      f"Test: {len(dataset_filtered['test'])}")

train_subset = dataset_filtered["train"].select(range(min(2000, len(dataset_filtered["train"]))))
val_subset   = dataset_filtered["validation"].select(range(min(100, len(dataset_filtered["validation"]))))
test_subset  = dataset_filtered["test"].select(range(min(100, len(dataset_filtered["validation"]))))

SYSTEM_PROMPT = (
    "Ti si AI asistent specijalizovan za sažimanje novinskih članaka "
    "na srpskom jeziku. Napravi kratak i informativan sažetak datog članka."
)

def format_example(example):
    """Формирует чат-сообщение для SFT"""
    article = clean_text(example["text"])
    summary = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Sažmi sledeći članak:\n\n{article}"},
        {"role": "assistant", "content": summary},
    ]
    return {"messages": messages}

train_dataset = train_subset.map(format_example)
val_dataset   = val_subset.map(format_example)
test_dataset  = test_subset.map(format_example)

print(f"Готово: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
print("\nПример messages[0]:", train_dataset[0]["messages"][1]["content"][:200])

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)

print(model.print_trainable_parameters())


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3_8b_serbian_sum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

trainer.train()


In [ ]:
model.save_pretrained("qwen3_8b_serbian_sum_lora")
tokenizer.save_pretrained("qwen3_8b_serbian_sum_lora")

import shutil
shutil.make_archive("/content/lora_adapter_8b", "zip", "/content/qwen3_8b_serbian_sum_lora")

from google.colab import files
files.download("/content/lora_adapter_8b.zip")
